<div dir="rtl" style="text-align: right;">

# اجرای اعتبارسنجی سؤالات از طریق مدل زبانی (LLM Validation Runner)

این نوت‌بوک فایل `validation_requests.json` (خروجی مرحله‌ی بازیابی معنایی) را می‌خواند، برای هر سؤال یک Prompt مبتنی بر شواهد واقعی نظرات می‌سازد، از طریق API سازگار با OpenAI (روی Metis AI) پاسخ می‌گیرد و نتایج را برای بررسی دستی در قالب JSON و CSV ذخیره می‌کند.

این نسخه علاوه بر موارد قبلی، **مصرف توکن و هزینه را ردیابی می‌کند** و یک **سقف بودجه (Budget Guard)** دارد: قبل از هر فراخوانی هزینه‌ی تجمعی چک می‌شود و در صورت رسیدن به سقف، اجرا متوقف می‌شود.

> ⚠️ **نکته درباره‌ی نرخ‌های هزینه:** مقادیر `PRICING_PER_1M_TOKENS` در بخش تنظیمات، بر مبنای قیمت‌گذاری عمومی OpenAI هستند و صرفاً **Placeholder** محسوب می‌شوند — چون Metis AI یک reseller ایرانی است و نرخ واقعی آن (و واحد پولی‌اش) ممکن است متفاوت باشد. قبل از تکیه‌کردن روی این اعداد برای تصمیم واقعی، آن‌ها را با نرخ دقیق حساب Metis خودتان جایگزین کنید.


</div>

<div dir="rtl" style="text-align: right;">


## بخش ۱ — تست اتصال به API و مشاهده‌ی مدل‌های در دسترس
</div>

In [ ]:
import requests

# اندپوینت لیست مدل‌ها (برای تست اتصال و اعتبار کلید)
API_URL = "https://api.metisai.ir/openai/v1/models"
API_KEY = "****"

headers = {"Authorization": f"Bearer {API_KEY}"}

response = requests.get(API_URL, headers=headers, timeout=30)

print("Status code:", response.status_code)
print("Response:")
print(response.text)


Status code: 200
Response:
{"object":"list","data":[{"id":"text-embedding-ada-002","object":"model","created":1671217299,"owned_by":"openai-internal"},{"id":"whisper-1","object":"model","created":1677532384,"owned_by":"openai-internal"},{"id":"gpt-3.5-turbo","object":"model","created":1677610602,"owned_by":"openai"},{"id":"tts-1","object":"model","created":1681940951,"owned_by":"openai-internal"},{"id":"gpt-3.5-turbo-16k","object":"model","created":1683758102,"owned_by":"openai-internal"},{"id":"gpt-4-0613","object":"model","created":1686588896,"owned_by":"openai"},{"id":"gpt-4","object":"model","created":1687882411,"owned_by":"openai"},{"id":"davinci-002","object":"model","created":1692634301,"owned_by":"system"},{"id":"babbage-002","object":"model","created":1692634615,"owned_by":"system"},{"id":"gpt-3.5-turbo-instruct","object":"model","created":1692901427,"owned_by":"system"},{"id":"gpt-3.5-turbo-instruct-0914","object":"model","created":1694122472,"owned_by":"system"},{"id":"gpt-3

<div dir="rtl" style="text-align: right;">

## بخش ۲ — تنظیمات، بارگذاری درخواست‌ها و بودجه‌ی مجاز

</div>

In [3]:
import json
import time
import requests
import pandas as pd

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

INPUT_FILE = "validation_requests.json"
OUTPUT_JSON = "validation_results.json"
OUTPUT_CSV = "validation_manual_review.csv"

# اندپوینت chat/completions
API_URL = "https://api.metisai.ir/openai/v1/chat/completions"
API_KEY = "tpsg-2gs5I7ZZP9uiGuN82wmv3brAnoIAV50"

MODEL_NAME = "gpt-4o-mini"

# ------------------------------------------------------------
# قیمت‌گذاری و بودجه (Cost Tracking & Budget Guard)
# ------------------------------------------------------------

# نرخ‌ها بر اساس قیمت‌گذاری عمومی OpenAI برای هر ۱ میلیون توکن است.
# این‌ها Placeholder هستند — قبل از اجرای واقعی با نرخ دقیق حساب
# Metis AI خودتان جایگزین کنید (ممکن است بر مبنای تومان و/یا نرخ متفاوت باشد).
PRICING_PER_1M_TOKENS = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4o": {"input": 2.50, "output": 10.00},
}

MAX_BUDGET_USD = 5.0  # سقف هزینه‌ی مجاز برای کل اجرای این نوت‌بوک

total_prompt_tokens = 0
total_completion_tokens = 0
total_cost_usd = 0.0


def estimate_cost(model, prompt_tokens, completion_tokens):
    rates = PRICING_PER_1M_TOKENS.get(model)
    if rates is None:
        return 0.0
    input_cost = (prompt_tokens / 1_000_000) * rates["input"]
    output_cost = (completion_tokens / 1_000_000) * rates["output"]
    return input_cost + output_cost


# ------------------------------------------------------------
# بارگذاری درخواست‌ها
# ------------------------------------------------------------

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    validation_requests = json.load(f)

print("Validation requests:", len(validation_requests))
print(f"Budget limit for this run: ${MAX_BUDGET_USD:.2f}")


Validation requests: 192
Budget limit for this run: $5.00


<div dir="rtl" style="text-align: right;">

## بخش ۳ — ساخت Prompt بر اساس شواهد (Evidence-grounded Prompt)

</div>

In [4]:
def build_prompt(item):
    product_title = item["product_title"]
    question = item["question"]

    evidence_text = []
    for evidence in item["evidence"]:
        evidence_text.append(f"""
COMMENT ID: {evidence["comment_id"]}

Title:
{evidence["title"]}

Body:
{evidence["body"]}

Advantages:
{evidence["advantages"]}

Disadvantages:
{evidence["disadvantages"]}

Rate:
{evidence["rate"]}

Recommendation:
{evidence["recommendation_status"]}
""")
    evidence_text = "\n\n".join(evidence_text)

    prompt = f"""
شما یک دستیار تحلیل نظرات کاربران دیجی‌کالا هستید.

محصول:
{product_title}

سؤال:
{question}

در ادامه تعدادی نظر واقعی کاربران درباره همین محصول ارائه شده است.

فقط و فقط بر اساس این نظرات پاسخ بده.

اگر شواهد کافی برای پاسخ وجود ندارد، صریحاً بگو که
شواهد کافی در نظرات ارائه‌شده وجود ندارد.

هیچ اطلاعاتی را که در شواهد وجود ندارد به عنوان واقعیت بیان نکن.

در پاسخ، در صورت امکان به Comment IDهای مرتبط اشاره کن.

نظرات:

{evidence_text}
"""
    return prompt


<div dir="rtl" style="text-align: right;">

## بخش ۴ — تابع فراخوانی API (با استخراج مصرف توکن)

</div>

In [5]:
def call_api(prompt):
    headers = {"Content-Type": "application/json"}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": "You are a grounded product-review question answering assistant.",
            },
            {"role": "user", "content": prompt},
        ],
        "temperature": 0,
    }

    start_time = time.perf_counter()
    response = requests.post(API_URL, headers=headers, json=payload, timeout=300)
    latency = time.perf_counter() - start_time

    response.raise_for_status()
    data = response.json()

    # پاسخ سازگار با فرمت OpenAI
    answer = data["choices"][0]["message"]["content"]

    # اطلاعات مصرف توکن (اگر API آن‌ها را برنگرداند، صفر در نظر گرفته می‌شود)
    usage = data.get("usage", {})
    prompt_tokens = usage.get("prompt_tokens", 0)
    completion_tokens = usage.get("completion_tokens", 0)

    return answer, latency, prompt_tokens, completion_tokens


<div dir="rtl" style="text-align: right;">

## بخش ۵ — اجرای اعتبارسنجی روی همه‌ی درخواست‌ها (با Budget Guard)

قبل از هر فراخوانی، هزینه‌ی تجمعی تا این لحظه با `MAX_BUDGET_USD` مقایسه می‌شود. به‌محض رسیدن به سقف، حلقه بدون فراخوانی موارد باقی‌مانده متوقف می‌شود (نه فقط هشدار) و موارد پردازش‌نشده با وضعیت `skipped_budget_limit` ثبت می‌شوند تا در خروجی نهایی هم مشخص باشند.

</div>

In [6]:
results = []

for index, item in enumerate(validation_requests, start=1):
    if total_cost_usd >= MAX_BUDGET_USD:
        print(
            f"\n🛑 Budget limit reached (${total_cost_usd:.4f} >= ${MAX_BUDGET_USD:.2f}). "
            f"Stopping before item {index}/{len(validation_requests)}."
        )
        results.append({
            "product_id": item["product_id"],
            "product_title": item["product_title"],
            "question": item["question"],
            "answer": None,
            "error": "skipped_budget_limit",
            "latency_seconds": None,
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "cost_usd": 0.0,
            "evidence": item["evidence"],
        })
        continue

    print(f"[{index}/{len(validation_requests)}] {item['product_title']} | {item['question']}")

    prompt = build_prompt(item)

    try:
        answer, latency, prompt_tokens, completion_tokens = call_api(prompt)
        call_cost = estimate_cost(MODEL_NAME, prompt_tokens, completion_tokens)

        total_prompt_tokens += prompt_tokens
        total_completion_tokens += completion_tokens
        total_cost_usd += call_cost

        result = {
            "product_id": item["product_id"],
            "product_title": item["product_title"],
            "question": item["question"],
            "answer": answer,
            "latency_seconds": latency,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "cost_usd": call_cost,
            "evidence": item["evidence"],
        }
        results.append(result)

        print(
            f"Latency: {latency:.2f}s | tokens: {prompt_tokens}+{completion_tokens} "
            f"| cost: ${call_cost:.4f} | running total: ${total_cost_usd:.4f}"
        )

    except Exception as e:
        print("ERROR:", str(e))
        results.append({
            "product_id": item["product_id"],
            "product_title": item["product_title"],
            "question": item["question"],
            "answer": None,
            "error": str(e),
            "latency_seconds": None,
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "cost_usd": 0.0,
            "evidence": item["evidence"],
        })


[1/192] برس مژه و ابرو کد BR01ABO | مردم بیشتر از چه چیزی در این محصول راضی بودند؟
Latency: 2.79s | tokens: 461+125 | cost: $0.0001 | running total: $0.0001
[2/192] برس مژه و ابرو کد BR01ABO | ایرادهای پرتکرار این محصول چیست؟
Latency: 2.43s | tokens: 478+41 | cost: $0.0001 | running total: $0.0002
[3/192] برس مژه و ابرو کد BR01ABO | خریداران درباره‌ی کیفیت این محصول چه گفته‌اند؟
Latency: 2.86s | tokens: 479+141 | cost: $0.0002 | running total: $0.0004
[4/192] برس مژه و ابرو کد BR01ABO | آیا با توجه به تجربه‌ی کاربران ارزش خرید دارد؟
Latency: 2.91s | tokens: 445+233 | cost: $0.0002 | running total: $0.0006
[5/192] پیلینگ صورت مدل Eli-74 | مردم بیشتر از چه چیزی در این محصول راضی بودند؟
Latency: 2.85s | tokens: 458+126 | cost: $0.0001 | running total: $0.0007
[6/192] پیلینگ صورت مدل Eli-74 | ایرادهای پرتکرار این محصول چیست؟
Latency: 2.53s | tokens: 465+41 | cost: $0.0001 | running total: $0.0008
[7/192] پیلینگ صورت مدل Eli-74 | خریداران درباره‌ی کیفیت این محصول چه گفته‌اند؟
Latency: 3.87s

<div dir="rtl" style="text-align: right;">

## بخش ۶ — خلاصه‌ی مصرف توکن و هزینه

</div>

In [7]:
print("=" * 40)
print(f"Total prompt tokens:     {total_prompt_tokens}")
print(f"Total completion tokens: {total_completion_tokens}")
print(f"Total tokens:            {total_prompt_tokens + total_completion_tokens}")
print(f"Estimated total cost:    ${total_cost_usd:.4f}")
print(f"Budget limit:            ${MAX_BUDGET_USD:.2f}")
print(f"Remaining budget:        ${MAX_BUDGET_USD - total_cost_usd:.4f}")
print("=" * 40)


Total prompt tokens:     97884
Total completion tokens: 31097
Total tokens:            128981
Estimated total cost:    $0.0333
Budget limit:            $5.00
Remaining budget:        $4.9667


<div dir="rtl" style="text-align: right;">

## بخش ۷ — ذخیره‌سازی نتایج خام (JSON)

</div>

In [8]:
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Saved JSON: {OUTPUT_JSON}")


Saved JSON: validation_results.json


<div dir="rtl" style="text-align: right;">

## بخش ۸ — ساخت CSV نهایی برای بررسی دستی (Manual Review)

این جدول برای هر سؤال، پاسخ مدل، هزینه‌ی همان فراخوانی و متن کامل شواهد را کنار هم می‌گذارد.

</div>

In [9]:
csv_rows = []

for result in results:
    evidence_ids = []
    evidence_texts = []

    for evidence in result["evidence"]:
        evidence_ids.append(str(evidence["comment_id"]))

        text = (
            f"[COMMENT ID: {evidence['comment_id']}]\n"
            f"Title: {evidence['title']}\n"
            f"Body: {evidence['body']}\n"
            f"Advantages: {evidence['advantages']}\n"
            f"Disadvantages: {evidence['disadvantages']}"
        )
        evidence_texts.append(text)

    csv_rows.append({
        "product_id": result["product_id"],
        "product_title": result["product_title"],
        "question": result["question"],
        "api_answer": result["answer"],
        "prompt_tokens": result.get("prompt_tokens", 0),
        "completion_tokens": result.get("completion_tokens", 0),
        "cost_usd": result.get("cost_usd", 0.0),
        "evidence_comment_ids": " | ".join(evidence_ids),
        "evidence_comments": "\n\n".join(evidence_texts),
        "latency_seconds": result["latency_seconds"],
    })

df_validation = pd.DataFrame(csv_rows)
df_validation.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"Saved: {OUTPUT_CSV}")
print("\nRows:", len(df_validation))


Saved: validation_manual_review.csv

Rows: 192
